# 04 — Evaluation, SHAP & Business Read
**RetainIQ India (Day 8).** Evaluation *beyond accuracy*, driver explanations (SHAP + odds ratios), and the stakeholder read. Every figure is produced by the tested `src/` modules, so the notebook and pipeline never diverge.

In [1]:
import sys, pandas as pd
sys.path.insert(0,'../src')
import evaluate, explain

## 1. Evaluation beyond accuracy
Accuracy is misleading on a 26.5%-churn base. We judge the model on **ranking** (ROC-AUC, PR-AUC), **probability quality** (Brier + reliability), and **business lift**.

In [2]:
ev = evaluate.run()
print('ROC-AUC :', ev['headline']['roc_auc'])
print('PR-AUC  :', ev['headline']['pr_auc'])
print('Brier   :', ev['headline']['brier'])
print('Calibration gap (max, deciles):', ev['max_calibration_gap'])
print('F1-max threshold:', ev['f1_max_threshold'])
print('Top-3 decile capture:', ev['top3_decile_capture'])

05:48:55 | INFO    | model | leakage-free split: train=5634 test=1409 (stratified, seed=42)


05:48:55 | INFO    | model | Logistic (calibrated): {'roc_auc': 0.8441, 'pr_auc': 0.6595, 'brier': 0.1367, 'accuracy': 0.7999, 'precision': 0.6554, 'recall': 0.5187, 'f1': 0.5791}


05:48:55 | INFO    | model | Brier raw=0.1367 -> calibrated=0.1367


05:48:57 | INFO    | model | RandomForest (challenger): {'roc_auc': 0.846, 'pr_auc': 0.6705, 'brier': 0.1357, 'accuracy': 0.7999, 'precision': 0.6523, 'recall': 0.5267, 'f1': 0.5828}


05:48:57 | INFO    | model | saved primary model -> models/churn_model.pkl


...

05:48:57 | INFO    | model | F1-max threshold = 0.30 (P=0.530 R=0.757 F1=0.623)


05:48:57 | INFO    | model | Decile lift / cumulative capture:
 decile  customers  churners  churn_rate  lift  cum_churners  cum_capture
      1        141       107    0.758865  2.86           107        0.286
      2        141        80    0.567376  2.14           187        0.500
      3        141        56    0.397163  1.50           243        0.650
      4        141        47    0.333333  1.26           290        0.775
      5        141        39    0.276596  1.04           329        0.880
      6        141        19    0.134752  0.51           348        0.930
      7        141        18    0.127660  0.48           366        0.979
      8        141         5    0.035461  0.13           371        0.992
      9        141         2    0.014184  0.05           373        0.997
     10        140         1    0.007143  0.03           374        1.000


05:48:57 | INFO    | model | Top-3 deciles (30% of base) capture 65.0% of all churners


05:48:57 | INFO    | model | saved reports/evaluation.json + 3 figures


ROC-AUC : 0.8441
PR-AUC  : 0.6595
Brier   : 0.1367
Calibration gap (max, deciles): 0.0554
F1-max threshold: {'threshold': 0.3, 'precision': 0.53, 'recall': 0.7567, 'f1': 0.6233}
Top-3 decile capture: 0.65


### Decile lift / cumulative capture
Sort customers by predicted risk, split into deciles. The top deciles concentrate churners — this is what makes budget-constrained targeting possible.

In [3]:
pd.DataFrame(ev['decile_lift'])[['decile','customers','churners','churn_rate','lift','cum_capture']]

,decile,customers,churners,churn_rate,lift,cum_capture
0,1,141,107,0.758865,2.86,0.286
1,2,141,80,0.567376,2.14,0.500
2,3,141,56,0.397163,1.50,0.650
3,4,141,47,0.333333,1.26,0.775
4,5,141,39,0.276596,1.04,0.880
5,6,141,19,0.134752,0.51,0.930
6,7,141,18,0.127660,0.48,0.979
7,8,141,5,0.035461,0.13,0.992
8,9,141,2,0.014184,0.05,0.997
9,10,140,1,0.007143,0.03,1.000


![ROC & PR](../reports/figures/roc_pr_curves.png)
![Reliability](../reports/figures/calibration_curve.png)
![Threshold sweep](../reports/figures/threshold_sweep.png)

## 2. Why the model flags a customer — SHAP + odds ratios
We explain the underlying logistic (calibration is monotonic, so driver direction/ranking is preserved). SHAP and coefficients agree.

In [4]:
ex = explain.run()
pd.DataFrame(ex['coef_drivers']).head(10)

05:48:58 | INFO    | model | Top drivers by |coef| (odds ratio = multiplicative effect on churn odds):


05:48:58 | INFO    | model |    tenure_months                          coef=-1.375  OR=0.253


05:48:58 | INFO    | model |    services_held                          coef=+0.874  OR=2.396


05:48:58 | INFO    | model |    contract_type = Two year               coef=-0.848  OR=0.428


05:48:58 | INFO    | model |    internet_type = No                     coef=-0.830  OR=0.436


...

05:48:59 | INFO    | model | Top drivers by mean |SHAP|: tenure_months, services_held, protection_count, total_charges_inr, internet_type = Fiber optic, contract_type = Two year


05:48:59 | INFO    | model | RAISES churn: services_held (×2.40 odds); total_charges_inr (×1.92 odds); internet_type = Fiber optic (×1.88 odds); contract_type = Month-to-month (×1.72 odds); paperless_billing (×1.45 odds)


05:48:59 | INFO    | model | LOWERS churn: tenure_months (×0.25 odds); contract_type = Two year (×0.43 odds); internet_type = No (×0.44 odds); has_phone (×0.44 odds); protection_count (×0.51 odds)


05:48:59 | INFO    | model | saved reports/shap_drivers.json + shap_summary.png + shap_bar.png


,feature,coef,odds_ratio
0,tenure_months,-1.375,0.253
1,services_held,0.874,2.396
2,contract_type = Two year,-0.848,0.428
3,internet_type = No,-0.830,0.436
4,has_phone,-0.828,0.437
5,protection_count,-0.681,0.506
6,total_charges_inr,0.655,1.924
7,internet_type = Fiber optic,0.629,1.876
8,contract_type = Month-to-month,0.540,1.716
9,paperless_billing,0.368,1.445


![SHAP beeswarm](../reports/figures/shap_summary.png)
![SHAP bar](../reports/figures/shap_bar.png)

### Plain-English drivers

In [5]:
import json; print(json.dumps(ex['narrative'], indent=2))

{
  "raises_churn": [
    "services_held (\u00d72.40 odds)",
    "total_charges_inr (\u00d71.92 odds)",
    "internet_type = Fiber optic (\u00d71.88 odds)",
    "contract_type = Month-to-month (\u00d71.72 odds)",
    "paperless_billing (\u00d71.45 odds)"
  ],
  "lowers_churn": [
    "tenure_months (\u00d70.25 odds)",
    "contract_type = Two year (\u00d70.43 odds)",
    "internet_type = No (\u00d70.44 odds)",
    "has_phone (\u00d70.44 odds)",
    "protection_count (\u00d70.51 odds)"
  ]
}


**Honest caveat.** These are *conditional* effects; correlated inputs (e.g. `total_charges` with tenure) shouldn't be read in isolation. The decision-grade story — agreed by SHAP, coefficients, and the Day-6 marginal tests — is **tenure, contract type, and protection adoption**.

**Business takeaway.** Top 3 risk deciles hold ~65% of churners → concentrate budget there; lead with protection-bundle offers. Full stakeholder read: `docs/model_business_read.md`. The operating threshold is set by the Day 9-10 profit curve, not by F1.